# GuardianShield — Model Training & Evaluation

**Notebook 03:** เปรียบเทียบ Models และ Train VotingClassifier Ensemble

---
**Pipeline:**
1. Load preprocessed data
2. Build feature pipeline
3. Train & compare: LR, RF, GB, Voting
4. Evaluate on test set
5. Visualize confusion matrix & metrics


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

from src.data.generator import generate_dataset
from src.data.preprocessor import ThaiTextPreprocessor, preprocess_dataframe
from src.features.feature_engineering import build_feature_pipeline
from src.models.evaluator import compute_metrics, get_classification_report, get_confusion_matrix

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('Ready ✓')

## 1. Load & Preprocess Data

In [ ]:
from sklearn.model_selection import train_test_split

# Generate dataset
df = generate_dataset(n_ham=2500, n_spam=2000, n_phishing=1500, random_state=42)

# Preprocess
preprocessor = ThaiTextPreprocessor(remove_stopwords=True, engine='newmm')
df = preprocess_dataframe(df, preprocessor=preprocessor)

# Split
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(f'\nTrain label distribution:')
print(train_df['label_name'].value_counts())

## 2. Train & Compare Models

In [ ]:
import time

def build_and_evaluate(name, classifier, train_df, val_df):
    feature_pipe = build_feature_pipeline()
    pipeline = Pipeline([('features', feature_pipe), ('classifier', classifier)])
    
    start = time.time()
    pipeline.fit(train_df, train_df['label'])
    train_time = time.time() - start
    
    val_pred = pipeline.predict(val_df)
    val_proba = pipeline.predict_proba(val_df)
    metrics = compute_metrics(val_df['label'].values, val_pred, val_proba)
    
    return pipeline, metrics, train_time

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
}

results = {}
trained_models = {}

for name, clf in models.items():
    print(f'Training {name}...', end=' ')
    pipeline, metrics, train_time = build_and_evaluate(name, clf, train_df, val_df)
    results[name] = {'metrics': metrics, 'train_time': train_time}
    trained_models[name] = pipeline
    print(f'Acc={metrics["accuracy"]:.3f}, F1={metrics["f1_weighted"]:.3f}, Time={train_time:.1f}s')

## 3. Train Voting Ensemble

In [ ]:
# Build individual pipelines for VotingClassifier
estimator_list = []
for name, clf in models.items():
    feat = build_feature_pipeline()
    pipe = Pipeline([('features', feat), ('classifier', clf)])
    estimator_list.append((name.lower().replace(' ', '_'), pipe))

voter = VotingClassifier(estimators=estimator_list, voting='soft', weights=[2, 1, 2])

print('Training VotingClassifier...', end=' ')
start = time.time()
voter.fit(train_df, train_df['label'])
ens_time = time.time() - start

ens_pred = voter.predict(val_df)
ens_proba = voter.predict_proba(val_df)
ens_metrics = compute_metrics(val_df['label'].values, ens_pred, ens_proba)

results['Voting Ensemble'] = {'metrics': ens_metrics, 'train_time': ens_time}
print(f'Acc={ens_metrics["accuracy"]:.3f}, F1={ens_metrics["f1_weighted"]:.3f}, AUC={ens_metrics.get("auc_roc_weighted", 0):.3f}')

## 4. Model Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': v['metrics']['accuracy'],
        'F1 Weighted': v['metrics']['f1_weighted'],
        'AUC-ROC': v['metrics'].get('auc_roc_weighted', 0),
        'Train Time (s)': v['train_time'],
    }
    for name, v in results.items()
]).sort_values('F1 Weighted', ascending=False)

print('Model Comparison (Validation Set):')
print(comparison_df.to_string(index=False, float_format='{:.4f}'.format))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(comparison_df))
width = 0.25
ax.bar([xi - width for xi in x], comparison_df['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x, comparison_df['F1 Weighted'], width, label='F1 Weighted', alpha=0.8)
ax.bar([xi + width for xi in x], comparison_df['AUC-ROC'], width, label='AUC-ROC', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=15)
ax.set_ylim(0.7, 1.02)
ax.legend()
ax.set_title('Model Comparison — Validation Set')
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Confusion Matrix (Best Model)

In [ ]:
# Final evaluation on test set
test_pred = voter.predict(test_df)
cm = get_confusion_matrix(test_df['label'].values, test_pred)
test_metrics = compute_metrics(test_df['label'].values, test_pred, voter.predict_proba(test_df))

print('=== Final Test Set Results (VotingClassifier Ensemble) ===')
print(get_classification_report(test_df['label'].values, test_pred))

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['ham', 'spam', 'phishing'],
    yticklabels=['ham', 'spam', 'phishing'],
    ax=ax
)
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTest Accuracy:  {test_metrics["accuracy"]:.4f}')
print(f'Test F1 Score:  {test_metrics["f1_weighted"]:.4f}')
print(f'Test AUC-ROC:   {test_metrics.get("auc_roc_weighted", 0):.4f}')

## 6. Save Best Model

In [ ]:
import json

Path('../models').mkdir(exist_ok=True)
joblib.dump(voter, '../models/best_model.joblib')

metadata = {
    'model_name': 'VotingClassifier',
    'version': '1.0.0',
    'trained_at': pd.Timestamp.now().isoformat(),
    'val_accuracy': float(ens_metrics['accuracy']),
    'val_f1_weighted': float(ens_metrics['f1_weighted']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_f1_weighted': float(test_metrics['f1_weighted']),
    'test_auc_roc': float(test_metrics.get('auc_roc_weighted', 0)),
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_df),
    'label_names': ['ham', 'spam', 'phishing'],
    'model_path': '../models/best_model.joblib',
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Model saved to models/best_model.joblib')
print('Metadata saved to models/model_metadata.json')
print(f'\nFinal Model Summary:')
for k, v in metadata.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    elif not isinstance(v, list):
        print(f'  {k}: {v}')